In [1]:
import sys
import os
import torch
import yaml
from tqdm import tqdm
from torch.utils.data import DataLoader

sys.path.insert(0, '/home/jingjie/AutoTorch/src')

from data.idfraud.dataset import IDFraudTorchDataset
from data.idfraud.transforms import get_transform
from models import build_classifier_model, load_weights_from_checkpoint
from models.dino import load_dino_model
import pandas as pd
from omegaconf import OmegaConf

In [2]:
df = pd.read_csv('processed_recapture_tamper_feb.csv')

df['absolute_ori_path'] = df['absolute_ori_path'].str.replace('/routine_data/', '/routine_data_jj_feb/')
df['absolute_ocr_path'] = df['absolute_ocr_path'].str.replace('/routine_data/', '/routine_data_jj_feb/')

df.shape

(5596, 23)

In [3]:
# Config paths
ORI_EXP = "/home/jingjie/AutoTorch/runs/Ex2_dinov3_convnext_large_v1_512_ori"
CROP_EXP = "/home/jingjie/AutoTorch/runs/Ex2_dinov3_convnext_large_v1_512_crop"
BEST_CKPT_ORI = 2   # checkpoint for ORI model
BEST_CKPT_CROP = 11  # checkpoint for CROP model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def get_config(name: str = "/config.yaml"):
    cfg = OmegaConf.load(ORI_EXP + name)
    # Add computed values
    cfg.run_dir = f"{cfg.experiment.save_dir}/{cfg.experiment.save_name}"
    return cfg

cfg = get_config()
# Override head_type to match checkpoint (was trained with v1)
cfg.model.head_type = 'v1'
transform = get_transform(cfg.transform)
backbone,backbone_dim = load_dino_model(model_name=cfg.model.backbone_name)

/home/jingjie/.conda/envs/vision/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def run_inference(df, path_col, gt_col, exp_dir, ckpt_num, backbone, backbone_dim, transform, device, batch_size=32):
    """Run inference using specified path column and checkpoint."""
    # Prepare df with 'path' column
    df_temp = df.copy()
    df_temp['path'] = df_temp[path_col]
    
    # Create dataset and dataloader
    dataset = IDFraudTorchDataset(df_temp, transform=transform, gt_label=gt_col, img_path=path_col)
    dataloader = DataLoader(dataset, batch_size=batch_size, num_workers=8, pin_memory=True)
    
    # Load model
    ckpt_path = os.path.join(exp_dir, "checkpoints", f"epoch_{ckpt_num}.pt")
    model = build_classifier_model(device=device, backbone_model=backbone, input_dim=backbone_dim)
    model = load_weights_from_checkpoint(model, ckpt_path, device)
    model.eval()
    
    # Run inference
    all_probs = []
    with torch.inference_mode():
        for X, _ in tqdm(dataloader, desc=f"Inference {os.path.basename(exp_dir)}"):
            logits = model(X.to(device)).squeeze(1)
            all_probs.extend(torch.sigmoid(logits).cpu().tolist())
    
    return all_probs

In [ ]:
# df['exp2point5_pred_ori'] = run_inference(df=df, 
#                                         path_col='absolute_ori_path', 
#                                         gt_col='annot_is_idfraud',
#                                         exp_dir=ORI_EXP, 
#                                         ckpt_num=BEST_CKPT_ORI, 
#                                         backbone=backbone, 
#                                         backbone_dim=backbone_dim, 
#                                         transform=transform, 
#                                         device=device)

df['exp2point5_pred_crop'] = run_inference(df=df, 
                                        path_col='absolute_ocr_path', 
                                        gt_col='annot_is_idfraud',
                                        exp_dir=CROP_EXP, 
                                        ckpt_num=BEST_CKPT_CROP, 
                                        backbone=backbone, 
                                        backbone_dim=backbone_dim, 
                                        transform=transform, 
                                        device=device)





Inference Ex2_dinov3_convnext_large_v1_512_crop: 100%|██████████| 175/175 [01:14<00:00,  2.36it/s]


In [1]:

df['exp2point5_ori_label'] = (df['exp2point5_pred_ori'] > 0.5).astype(int)
df['exp2point5_crop_label'] = (df['exp2point5_pred_crop'] > 0.5).astype(int)

NameError: name 'df' is not defined

In [ ]:
df.columns

In [2]:
def calculate_apcer_bpcer(df, pred_col='parallel_label', gt_col='annot_isfraud'):
    """
    Calculate APCER and BPCER for binary classification.
    
    APCER = FN / (TP + FN) = attacks misclassified as genuine / total attacks
    BPCER = FP / (TN + FP) = genuine misclassified as attacks / total genuine
    
    Args:
        df: DataFrame with predictions and ground truth
        pred_col: Column name for model predictions (0=genuine, 1=attack)
        gt_col: Column name for ground truth labels
    """
    # Filter out unknown labels
    data = df[df[gt_col] != -1]
    
    # Ground truth
    attacks = data[data[gt_col] == 1]  # actual attacks (fraud)
    genuine = data[data[gt_col] == 0]  # actual genuine (bona fide)
    
    # APCER: attacks incorrectly classified as genuine
    fn = (attacks[pred_col] == 0).sum()  # false negatives
    apcer = fn / len(attacks) if len(attacks) > 0 else 0
    
    # BPCER: genuine incorrectly classified as attacks  
    fp = (genuine[pred_col] == 1).sum()  # false positives
    bpcer = fp / len(genuine) if len(genuine) > 0 else 0
    
    return {
        'APCER': apcer,
        'BPCER': bpcer,
        'Total Attacks': len(attacks),
        'Total Genuine': len(genuine),
        'FN (missed attacks)': fn,
        'FP (false alarms)': fp
    }


# Calculate for each model on df_exclude_quality
# print("=== Excluding Quality Issues ===")
# for model in ['crop_label', 'ori_label', 'parallel_label']:
#     metrics = calculate_apcer_bpcer(df_exclude_quality, model)
#     print(f"\n{model}:")
#     print(f"  APCER: {metrics['APCER']:.4f} ({metrics['FN (missed attacks)']}/{metrics['Total Attacks']})")
#     print(f"  BPCER: {metrics['BPCER']:.4f} ({metrics['FP (false alarms)']}/{metrics['Total Genuine']})")

# print("\n\n=== Including Quality Issues ===")
# for model in ['crop_label', 'ori_label', 'parallel_label']:
#     metrics = calculate_apcer_bpcer(df_not_exclude_quality, model)
#     print(f"\n{model}:")
#     print(f"  APCER: {metrics['APCER']:.4f} ({metrics['FN (missed attacks)']}/{metrics['Total Attacks']})")
#     print(f"  BPCER: {metrics['BPCER']:.4f} ({metrics['FP (false alarms)']}/{metrics['Total Genuine']})")

In [3]:
df['exp2point5_parallel_label'] = ((df['exp2point5_pred_ori'] + df['exp2point5_pred_crop']) / 2) > 0.5


NameError: name 'df' is not defined

In [38]:
import pandas as pd
df = pd.read_csv('/home/jingjie/AutoTorch/src/eval/idfraud/feb_results/exp2point5_results.csv')

In [39]:
import numpy as np
# Calculate average prediction
df['avg_pred'] = (df['exp2point5_pred_ori'] + df['exp2point5_pred_crop']) / 2

# Search thresholds
thresholds = np.arange(0.1, 0.9, 0.05)
results = []

for thresh in thresholds:
    df['temp_label'] = (df['avg_pred'] > thresh).astype(int)
    metrics = calculate_apcer_bpcer(df, pred_col='temp_label', gt_col='annot_is_idfraud')
    results.append({
        'threshold': thresh,
        'APCER': metrics['APCER'],
        'BPCER': metrics['BPCER'],
        'ACER': (metrics['APCER'] + metrics['BPCER']) / 2  # average error
    })

results_df = pd.DataFrame(results)
display(results_df)

# Find best threshold (lowest ACER)
best = results_df.loc[results_df['ACER'].idxmin()]
print(f"\nBest threshold: {best['threshold']:.2f} (APCER: {best['APCER']:.4f}, BPCER: {best['BPCER']:.4f})")

,threshold,APCER,BPCER,ACER
0,0.10,0.040323,0.302124,0.171223
1,0.15,0.048387,0.284028,0.166208
2,0.20,0.050403,0.272227,0.161315
3,0.25,0.054435,0.260228,0.157332
4,0.30,0.056452,0.249607,0.153029
5,0.35,0.056452,0.235051,0.145751
6,0.40,0.058468,0.220692,0.139580
7,0.45,0.060484,0.196105,0.128295
8,0.50,0.078629,0.140834,0.109732
9,0.55,0.102823,0.093430,0.098126



Best threshold: 0.75 (APCER: 0.1290, BPCER: 0.0545)


In [40]:
for model in ['exp2point5_ori_label','exp2point5_crop_label', 'exp2point5_parallel_label']:
    metrics = calculate_apcer_bpcer(df, pred_col=model, gt_col='annot_is_idfraud')
    print(f"\n{model}:")
    print(f"  APCER: {metrics['APCER']:.4f} ({metrics['FN (missed attacks)']}/{metrics['Total Attacks']})")
    print(f"  BPCER: {metrics['BPCER']:.4f} ({metrics['FP (false alarms)']}/{metrics['Total Genuine']})")


exp2point5_ori_label:
  APCER: 0.0927 (46/496)
  BPCER: 0.0765 (389/5084)

exp2point5_crop_label:
  APCER: 0.0887 (44/496)
  BPCER: 0.2364 (1202/5084)

exp2point5_parallel_label:
  APCER: 0.0786 (39/496)
  BPCER: 0.1408 (716/5084)


In [41]:
df_exclude_quality = df.loc[df['annot_is_Quality']==0]
df_exclude_quality.shape

(3330, 31)

In [42]:
df['annot_is_Quality'].value_counts()

annot_is_Quality
False    3330
True     2266
Name: count, dtype: int64

In [43]:
for model in ['exp2point5_ori_label','exp2point5_crop_label', 'exp2point5_parallel_label']:
    metrics = calculate_apcer_bpcer(df_exclude_quality, pred_col=model, gt_col='annot_is_idfraud')
    print(f"\n{model}:")
    print(f"  APCER: {metrics['APCER']:.4f} ({metrics['FN (missed attacks)']}/{metrics['Total Attacks']})")
    print(f"  BPCER: {metrics['BPCER']:.4f} ({metrics['FP (false alarms)']}/{metrics['Total Genuine']})")


exp2point5_ori_label:
  APCER: 0.0927 (46/496)
  BPCER: 0.0443 (125/2823)

exp2point5_crop_label:
  APCER: 0.0887 (44/496)
  BPCER: 0.1183 (334/2823)

exp2point5_parallel_label:
  APCER: 0.0786 (39/496)
  BPCER: 0.0670 (189/2823)


In [44]:

import ast 
# convert string representation of list to actual Python list
df["tag"] = df["tag"].apply(ast.literal_eval)
df_exclude_quality["tag"] = df_exclude_quality["tag"].apply(ast.literal_eval)

print(df["tag"].iloc[0])
print(type(df["tag"].iloc[0]))

['Genuine']
<class 'list'>


/tmp/ipykernel_473999/1948419258.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_exclude_quality["tag"] = df_exclude_quality["tag"].apply(ast.literal_eval)


In [45]:

def breakdown_by_tag(df, models=['exp2point5_ori_label','exp2point5_crop_label', 'exp2point5_parallel_label']):

    fraud_tags = [
            'Printed - Grayscale',
            'Printed - Color',
            'Cutout Printed - Grayscale',
            'Cutout Printed - Color',
            'Replay - Mobile',
            'Replay - Tablet',
            'Replay - Monitor',
            'Fullpage Printed - Color',
        ]
    
    rows = []
    for tag in fraud_tags + ['Genuine']:
        tag_df = df[df['tag'].apply(lambda x: tag in x)]
        if len(tag_df) == 0:
            continue
        row = {'tag': tag, 'count': len(tag_df)}
        for model in models:
            # For fraud: miss rate (pred=0), For genuine: false alarm rate (pred=1)
            # sum the preds==0 (model classifies as genuine when the tag is fraud) ; 
            error = (tag_df[model] == 0).sum() if tag != 'Genuine' else (tag_df[model] == 1).sum()
            row[model] = f"{error/len(tag_df):.2%} ({error}/{len(tag_df)})"
        rows.append(row)
    
    return pd.DataFrame(rows).set_index('tag')

# miss rate
print("=== Breakdown statistics: APCER for fraud and BPCER for genuine ===")
print("Exclude Quality images:")
display(breakdown_by_tag(df).reset_index())
print("All images:")
display(breakdown_by_tag(df_exclude_quality).reset_index())


=== Breakdown statistics: APCER for fraud and BPCER for genuine ===
Exclude Quality images:


,tag,count,exp2point5_ori_label,exp2point5_crop_label,exp2point5_parallel_label
0,Printed - Grayscale,17,5.88% (1/17),0.00% (0/17),0.00% (0/17)
1,Printed - Color,25,52.00% (13/25),64.00% (16/25),52.00% (13/25)
2,Cutout Printed - Grayscale,4,0.00% (0/4),0.00% (0/4),0.00% (0/4)
3,Cutout Printed - Color,34,38.24% (13/34),14.71% (5/34),17.65% (6/34)
4,Replay - Mobile,345,4.93% (17/345),6.09% (21/345),5.22% (18/345)
5,Replay - Tablet,6,16.67% (1/6),16.67% (1/6),16.67% (1/6)
6,Replay - Monitor,65,1.54% (1/65),1.54% (1/65),1.54% (1/65)
7,Genuine,5086,7.69% (391/5086),23.67% (1204/5086),14.12% (718/5086)


All images:


,tag,count,exp2point5_ori_label,exp2point5_crop_label,exp2point5_parallel_label
0,Printed - Grayscale,17,5.88% (1/17),0.00% (0/17),0.00% (0/17)
1,Printed - Color,25,52.00% (13/25),64.00% (16/25),52.00% (13/25)
2,Cutout Printed - Grayscale,4,0.00% (0/4),0.00% (0/4),0.00% (0/4)
3,Cutout Printed - Color,34,38.24% (13/34),14.71% (5/34),17.65% (6/34)
4,Replay - Mobile,345,4.93% (17/345),6.09% (21/345),5.22% (18/345)
5,Replay - Tablet,6,16.67% (1/6),16.67% (1/6),16.67% (1/6)
6,Replay - Monitor,65,1.54% (1/65),1.54% (1/65),1.54% (1/65)
7,Genuine,2825,4.50% (127/2825),11.89% (336/2825),6.76% (191/2825)


In [46]:
df.to_csv("feb_results/exp2point5_results.csv")